# 26.07 - Cross-role integration

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Pair-review notes containing five actionable comments and a validated CV-NLP fusion decision.

The goal is to support a teammate without silently breaking the CV data contract. You will inspect tensors and label mappings, review a deliberately flawed teammate configuration, and combine CV and NLP predictions using validation Macro-F1.


## Core Ideas

- A useful review comment names the problem, explains its consequence, and proposes a concrete fix.
- Cross-role integration needs an explicit contract for sample order, class order, tensor shape, dtype, and score type.
- Average probabilities only after both branches use the same row and class ordering.
- Choose a fusion weight on validation data; do not tune it on the test set.
- Macro-F1 is appropriate when every class should matter equally.


In [ ]:
import numpy as np
import torch
from sklearn.metrics import f1_score

SEED = 26
np.random.seed(SEED)
torch.manual_seed(SEED)


## Prepared Team Fixture

The fixture represents a small multimodal batch and a teammate configuration with several reviewable mistakes.

**Return structure — `make_team_fixture`:** Returns a `dict` with exactly these keys: `images`, a CPU `torch.float32` tensor shaped `[N, 3, 16, 16]`; `labels`, a CPU `torch.long` tensor shaped `[N]`; `cv_logits` and `nlp_logits`, CPU `torch.float32` tensors shaped `[N, C]`; `label_to_index`, a `dict[str, int]` containing exactly `C` contiguous indices; and `teammate_config`, a dictionary whose values are `bool` or `str`. Here `N=12` samples and `C=3` classes.


In [ ]:
def make_team_fixture(seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    labels = torch.tensor([0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 1, 2], dtype=torch.long)
    images = torch.rand((12, 3, 16, 16), generator=generator, dtype=torch.float32)
    cv_logits = torch.randn((12, 3), generator=generator) * 0.35
    nlp_logits = torch.randn((12, 3), generator=generator) * 0.35
    cv_logits[torch.arange(12), labels] += 1.35
    nlp_logits[torch.arange(12), labels] += 0.85
    nlp_logits[[1, 7], 1] -= 1.6
    cv_logits[[5, 11], 2] -= 1.8
    return {
        "images": images,
        "labels": labels,
        "cv_logits": cv_logits,
        "nlp_logits": nlp_logits,
        "label_to_index": {"calm": 0, "focus": 1, "stress": 2},
        "teammate_config": {
            "validation_shuffle": True,
            "calls_model_eval": False,
            "uses_inference_mode": False,
            "selection_metric": "accuracy",
            "class_order_source": "independent alphabetical sort",
        },
    }


fixture = make_team_fixture()
print("images:", fixture["images"].shape, fixture["images"].dtype)
print("CV/NLP logits:", fixture["cv_logits"].shape, fixture["nlp_logits"].shape)


## Exercise 26-A: Audit the CV batch contract

Check rank, channel count, dtype, finite values, label dtype/range, and whether the label mapping contains contiguous indices.

**Return structure — `audit_cv_contract`:** Returns a `dict` with exactly `image_shape` (`tuple[int, int, int, int]`), `image_dtype` (`str`), `label_shape` (`tuple[int]`), `label_dtype` (`str`), `mapping_ok` (`bool`), and `issues` (`list[str]`, possibly empty). The function does not mutate its inputs.


In [ ]:
# TODO 26-A
def audit_cv_contract(images, labels, label_to_index):
    raise NotImplementedError("TODO 26-A")


# Smoke check: run this after implementing the function above.
smoke_contract = audit_cv_contract(fixture["images"], fixture["labels"], fixture["label_to_index"])
print("contract issues:", smoke_contract["issues"])


## Exercise 26-B: Write five actionable review comments

Review the supplied teammate configuration. Each comment must identify a risk and a proposed fix.

**Return structure — `review_teammate_pipeline`:** Returns a `list[dict]` of exactly five items. Every item has exactly `area`, `risk`, and `fix`, each a non-empty `str`. The input dictionary is not modified.


In [ ]:
# TODO 26-B
def review_teammate_pipeline(config):
    raise NotImplementedError("TODO 26-B")


# Smoke check: run this after implementing the function above.
smoke_comments = review_teammate_pipeline(fixture["teammate_config"])
print("review comment count:", len(smoke_comments))


## Exercise 26-C: Fuse CV and NLP probabilities

Convert each branch's logits to probabilities, verify matching shapes, and compute a weighted average.

**Return structure — `fuse_modal_probabilities`:** Returns a tuple of two CPU tensors. Position 0 is a `torch.float32` probability tensor shaped `[N, C]` whose rows sum to one. Position 1 is a `torch.long` prediction tensor shaped `[N]`. The scalar `cv_weight` must be in `[0, 1]`.


In [ ]:
# TODO 26-C
def fuse_modal_probabilities(cv_logits, nlp_logits, cv_weight):
    raise NotImplementedError("TODO 26-C")


# Smoke check: run this after implementing the function above.
smoke_probs, smoke_predictions = fuse_modal_probabilities(
    fixture["cv_logits"], fixture["nlp_logits"], cv_weight=0.6
)
print("fused:", smoke_probs.shape, smoke_predictions.tolist())


## Exercise 26-D: Select a fusion weight on validation data

Evaluate every supplied weight with Macro-F1 and retain the first best weight to make ties deterministic.

**Return structure — `select_fusion_weight`:** Returns a `dict` with exactly `best_weight` (`float`), `best_macro_f1` (`float` in `[0,1]`), and `scores` (`list[dict]`). Each score item contains `cv_weight` and `macro_f1`, both `float`; list order matches `candidate_weights`.


In [ ]:
# TODO 26-D
def select_fusion_weight(cv_logits, nlp_logits, labels, candidate_weights):
    raise NotImplementedError("TODO 26-D")


# Smoke check: run this after implementing the function above.
smoke_selection = select_fusion_weight(
    fixture["cv_logits"], fixture["nlp_logits"], fixture["labels"], [0.0, 0.25, 0.5, 0.75, 1.0]
)
print("best fusion:", smoke_selection["best_weight"], smoke_selection["best_macro_f1"])


## Test Cases

Run this cell after completing all TODO cells. A correct implementation prints `Day 26 tests passed`.

**Return structure — `run_day26_tests`:** Returns `None`. Success is communicated by assertions completing and the exact printed message `Day 26 tests passed`.


In [ ]:
def run_day26_tests():
    contract = audit_cv_contract(fixture["images"], fixture["labels"], fixture["label_to_index"])
    assert set(contract) == {"image_shape", "image_dtype", "label_shape", "label_dtype", "mapping_ok", "issues"}
    assert contract["image_shape"] == (12, 3, 16, 16)
    assert contract["mapping_ok"] is True and contract["issues"] == []

    comments = review_teammate_pipeline(fixture["teammate_config"])
    assert isinstance(comments, list) and len(comments) == 5
    assert all(set(item) == {"area", "risk", "fix"} for item in comments)
    assert all(all(isinstance(item[key], str) and item[key] for key in item) for item in comments)

    probabilities, predictions = fuse_modal_probabilities(fixture["cv_logits"], fixture["nlp_logits"], 0.5)
    assert probabilities.shape == (12, 3) and probabilities.dtype == torch.float32
    assert predictions.shape == (12,) and predictions.dtype == torch.long
    assert torch.allclose(probabilities.sum(dim=1), torch.ones(12), atol=1e-6)

    candidates = [0.0, 0.25, 0.5, 0.75, 1.0]
    selection = select_fusion_weight(
        fixture["cv_logits"], fixture["nlp_logits"], fixture["labels"], candidates
    )
    assert set(selection) == {"best_weight", "best_macro_f1", "scores"}
    assert selection["best_weight"] in candidates
    assert len(selection["scores"]) == len(candidates)
    assert np.isclose(selection["best_macro_f1"], max(item["macro_f1"] for item in selection["scores"]))
    print("Day 26 tests passed")


run_day26_tests()


## Day 26 Checklist

- [ ] I checked tensor shape, dtype, label range, and mapping order.
- [ ] I wrote five comments with a risk and a concrete fix.
- [ ] I fused probabilities rather than raw logits.
- [ ] I selected the fusion weight only on validation data.
- [ ] I can explain how row-order or class-order mismatch corrupts multimodal fusion.
